# CeNNMixer-v4: context and generation quality repair

Defaults: quality run, 128/256/512-token contexts, no exploration bypass. Start fresh; the old 64-token smoke run is not evidence of usable context.

Fixes include contiguous corpus windows, a context-length curriculum, retrieval chat prefixes, frequent on-policy feedback, chunked vocabulary losses, mixer-aware warmup selection, strict generation/context gates, and a held-out test split at final conversion.

The recurrent core remains CeNN + compact gated delta memory. Full attention layers elsewhere in Qwen remain intact. Parameter savings refer only to the selected mixer and are not a measured speedup.

For 1024-token experiments set SEQ_LEN=1024 and CONTEXT_LENGTHS="128,256,512,1024". Longer context increases training memory/time and must pass the same tests. No 32K/128K quality claim is made.

Research rationale and limitations: [docs/CENNMIXER_V4_CONTEXT_QUALITY.md](https://github.com/vtavakkoli/TinyCeNN-LM/blob/fix/cenn-v4-context-quality/docs/CENNMIXER_V4_CONTEXT_QUALITY.md).


In [ ]:
#@title 1. Setup
import pathlib, subprocess, sys, importlib, json, torch, shutil

REPO_REF='fix/cenn-v4-context-quality'
REPO_DIR=pathlib.Path('/content/TinyCeNN-LM-v4-fixed')
if not REPO_DIR.exists():
    subprocess.run(['git','clone','--branch',REPO_REF,
                    'https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)],check=True)
else:
    dirty=subprocess.check_output(['git','-C',str(REPO_DIR),'status','--porcelain'],text=True)
    if dirty.strip():
        raise RuntimeError('Checkout has local edits. Preserve them or choose another REPO_DIR.')
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',REPO_REF],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout',REPO_REF],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'merge','--ff-only','FETCH_HEAD'],check=True)

subprocess.run([
    sys.executable,'-m','pip','install','-q','-U',
    'transformers==5.17.0','accelerate','datasets','pandas','matplotlib',
    'huggingface_hub','safetensors'
],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR)],check=True)

SRC=REPO_DIR/'src'
if str(SRC) not in sys.path:
    sys.path.insert(0,str(SRC))
for n in list(sys.modules):
    if n=='tinycenn_lm' or n.startswith('tinycenn_lm.'):
        del sys.modules[n]
importlib.invalidate_caches()

for p in [
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_cennmixer_v4_core.py',
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_cennmixer_v4.py',
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_cennmixer_v4_train.py',
    REPO_DIR/'scripts'/'run_qwen35_cennmixer_v4.py',
]:
    subprocess.run([sys.executable,'-m','py_compile',str(p)],check=True)

print('✓ CeNNMixer-v4 preflight OK')
print('CUDA:',torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then rerun setup.')


In [ ]:
#@title 2. Configuration
BASE_MODEL='Qwen/Qwen3.5-0.8B' #@param {type:'string'}
LAYERS='0' #@param {type:'string'}
QUICK_SMOKE=False #@param {type:'boolean'}
EXPLORE_HIGH_ALPHA=False #@param {type:'boolean'}
RESUME_ALPHA=-1.0 #@param {type:'number'}

# CeNN local branch
GROUPS=24 #@param {type:'integer'}
CELL_DIM=32 #@param {type:'integer'}
GRAPH_STEPS=1 #@param {type:'integer'}

# DeltaCell associative branch
ASSOC_HEADS=8 #@param {type:'integer'}
KEY_DIM=32 #@param {type:'integer'}
VALUE_DIM=64 #@param {type:'integer'}
CONV_KERNEL=4 #@param {type:'integer'}

LR=0.00005 #@param {type:'number'}
ON_POLICY_EVERY=4 #@param {type:'integer'}
ON_POLICY_TOKENS=32 #@param {type:'integer'}

# Quick mode reduces updates, but preserves context and quality thresholds.
ALPHAS='0,0.05,0.10,0.20,0.30,0.40,0.50,0.60,0.70,0.80,0.90,1.0' #@param {type:'string'}
CONTEXT_LENGTHS='128,256,512' #@param {type:'string'}
SEQ_LEN=512 #@param {type:'integer'}
TRAIN_BLOCKS=1536 #@param {type:'integer'}
VAL_BLOCKS=8 #@param {type:'integer'}
STAGE_UPDATES=300 #@param {type:'integer'}
EXTEND_UPDATES=200 #@param {type:'integer'}
MAX_STAGE_UPDATES=3000 #@param {type:'integer'}
PROBE_EVERY=50 #@param {type:'integer'}
PATIENCE_PROBES=8 #@param {type:'integer'}
MIN_LR=0.0000125 #@param {type:'number'}
TOPK=64 #@param {type:'integer'}

MIN_TOP1=0.97 #@param {type:'number'}
MAX_KL=0.03 #@param {type:'number'}
MAX_HIDDEN_MSE=0.05 #@param {type:'number'}
MAX_MIXER_MSE=0.12 #@param {type:'number'}

# Outside the checkout; set a mounted Drive path for persistence across runtime resets.
OUTPUT_DIR=pathlib.Path('/content/cennmixer_v4_context_quality_results')

print('Quick smoke:',QUICK_SMOKE)
print('Explore higher alpha after soft misses:',EXPLORE_HIGH_ALPHA)
print('Resume alpha (-1 means fresh run):',RESUME_ALPHA)
print('CeNN local:',GROUPS,'×',CELL_DIM)
print('DeltaCell state:',ASSOC_HEADS,'×',KEY_DIM,'×',VALUE_DIM)
print('Runtime associative state floats:',ASSOC_HEADS*KEY_DIM*VALUE_DIM)


### Quality-first training

Smoke mode changes the training budget only and can never certify the final model. The worst result across all configured validation contexts controls acceptance. Short factual and retrieval tasks must preserve answers the teacher gets right; empty and repetitive generations fail. Jaccard remains descriptive only.

Every stage receives at least 100 updates (40 in smoke mode), with all context lengths introduced during that minimum budget. Failed stages stop by default. An experimental exploration flag remains available but cannot bypass the generation check.


### Resume a saved stage

Set RESUME_ALPHA to a saved checkpoint in OUTPUT_DIR. The runner re-enters that same alpha to validate or retrain it before advancing. Weights resume; optimizer/RNG state and report history start fresh. Use a new output directory for a fresh run and a mounted Drive path for durable Colab checkpoints.


In [ ]:
#@title 3. Train CeNNMixer-v4
cmd=[
    sys.executable,'-u',str(REPO_DIR/'scripts'/'run_qwen35_cennmixer_v4.py'),
    '--base-model',BASE_MODEL,
    '--layers',LAYERS,
    '--alphas',ALPHAS,
    '--seq-len',str(SEQ_LEN),
    '--context-lengths',CONTEXT_LENGTHS,
    '--train-blocks',str(TRAIN_BLOCKS),
    '--val-blocks',str(VAL_BLOCKS),
    '--groups',str(GROUPS),
    '--cell-dim',str(CELL_DIM),
    '--graph-steps',str(GRAPH_STEPS),
    '--assoc-heads',str(ASSOC_HEADS),
    '--key-dim',str(KEY_DIM),
    '--value-dim',str(VALUE_DIM),
    '--conv-kernel',str(CONV_KERNEL),
    '--lr',str(LR),
    '--stage-updates',str(STAGE_UPDATES),
    '--extend-updates',str(EXTEND_UPDATES),
    '--max-stage-updates',str(MAX_STAGE_UPDATES),
    '--probe-every',str(PROBE_EVERY),
    '--patience-probes',str(PATIENCE_PROBES),
    '--min-lr',str(MIN_LR),
    '--topk',str(TOPK),
    '--on-policy-every',str(ON_POLICY_EVERY),
    '--on-policy-tokens',str(ON_POLICY_TOKENS),
    '--min-top1',str(MIN_TOP1),
    '--max-kl',str(MAX_KL),
    '--max-hidden-mse',str(MAX_HIDDEN_MSE),
    '--max-mixer-mse',str(MAX_MIXER_MSE),
    '--output-dir',str(OUTPUT_DIR),
]
if QUICK_SMOKE:
    cmd.append('--quick-smoke')
if EXPLORE_HIGH_ALPHA:
    cmd.append('--explore-high-alpha')
if RESUME_ALPHA >= 0:
    cmd += ['--resume-alpha',str(RESUME_ALPHA)]

print('='*120)
print(' '.join(cmd))
print('='*120)

p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in iter(p.stdout.readline,''):
    print(line,end='',flush=True)
rc=p.wait()
print('\nFinished, exit code',rc)
if rc:
    raise subprocess.CalledProcessError(rc,cmd)


In [ ]:
#@title 4. Results and parameter reduction
import pandas as pd
from IPython.display import display

report=json.loads((OUTPUT_DIR/'report.json').read_text())
stage=pd.read_csv(OUTPUT_DIR/'stage_summary.csv')
try:
    hist=pd.read_csv(OUTPUT_DIR/'training_history.csv')
except pd.errors.EmptyDataError:
    hist=pd.DataFrame()

print('Architecture:',report['architecture'])
print('Reached alpha:',report['reached_alpha'])
print('Last passing alpha:',report.get('last_passing_alpha'))
print('Evaluated context lengths:',report.get('evaluated_context_lengths'))
print('Progression complete:',report['progression_complete'])
print('Exploration mode:',report.get('exploration_mode'))
print('Exploration reached alpha:',report.get('exploration_reached_alpha'))
print('Strict final quality:',report['strict_quality_gate'])
print('CeNN-v4 params:',f"{report['cenn_params']:,}")
print('Qwen mixer params:',f"{report['replaced_qwen_mixer_params']:,}")
print('Mixer parameter reduction:',f"{report['mixer_param_reduction_pct']:.2f}%")

display(stage[[
    'alpha','best_step','trained_steps','pass','exploration_continue','violation',
    'student_ce','ce_gap','kl','reverse_kl','hidden_mse','delta_mse',
    'mixer_mse','mixer_cosine','mixer_delta','top1','top1_margin_loss',
    'generation_exact_rate','generation_mean_jaccard','generation_warning'
]])


In [ ]:
#@title 5. Compare directly with CeNNMixer-v3
V3_ALPHA06={
    'top1':0.84765625,
    'kl':0.1179187144,
    'hidden_mse':0.0986091695,
    'mixer_mse':0.1808171482,
    'generation_jaccard':0.1597098516,
}

print('Historical v3 smoke reference ONLY; different context/data, not a controlled comparison:',V3_ALPHA06)
for a in [0.5,0.6,0.7]:
    row=stage[(stage['alpha']-a).abs()<1e-9]
    if len(row):
        r=row.iloc[0]
        print(f"\nv4 alpha={a:g}")
        print(' top1:',float(r.top1))
        print(' KL:',float(r.kl))
        print(' hidden MSE:',float(r.hidden_mse))
        print(' mixer MSE:',float(r.mixer_mse))
        print(' generation Jaccard:',float(r.generation_mean_jaccard))

if len(stage[(stage['alpha']-0.6).abs()<1e-9]):
    r=stage[(stage['alpha']-0.6).abs()<1e-9].iloc[0]
    print('\nΔ v4-v3 at alpha=0.6:')
    print(' top1:',float(r.top1)-V3_ALPHA06['top1'])
    print(' generation Jaccard:',float(r.generation_mean_jaccard)-V3_ALPHA06['generation_jaccard'])


In [ ]:
#@title 6. Curves
import matplotlib.pyplot as plt

plt.figure(figsize=(9,5))
plt.plot(stage['alpha'],stage['top1'],marker='o')
plt.axhline(V3_ALPHA06['top1'],linestyle='--')
plt.xlabel('alpha')
plt.ylabel('top-1 agreement')
plt.title('CeNNMixer-v4 takeover — dashed line is v3 alpha=0.6 reference')
plt.show()

plt.figure(figsize=(9,5))
plt.plot(stage['alpha'],stage['generation_mean_jaccard'],marker='o')
plt.axhline(V3_ALPHA06['generation_jaccard'],linestyle='--')
plt.xlabel('alpha')
plt.ylabel('generation token-set Jaccard')
plt.title('Generation stability')
plt.show()

plt.figure(figsize=(9,5))
plt.plot(stage['alpha'],stage['mixer_mse'],marker='o',label='mixer MSE')
plt.plot(stage['alpha'],stage['mixer_cosine'],marker='o',label='cosine error')
plt.plot(stage['alpha'],stage['mixer_delta'],marker='o',label='temporal error')
plt.xlabel('alpha')
plt.ylabel('error')
plt.legend()
plt.title('Direct mixer fidelity')
plt.show()


### Interpreting acceptance

`pass=True` requires numerical fidelity at every configured context plus basic generation health. These small development probes do not establish broad chat quality. Final acceptance also requires a separate Wikitext test split, fresh retrieval keys, and a non-smoke run. Review the printed text before enabling upload.


In [ ]:
#@title 7. Final / latest generation samples
if report.get('progression_complete'):
    rows=report['final_cenn_only_generation']
    print('Final CeNN-only generation:')
else:
    last=float(stage.iloc[-1]['alpha'])
    path=OUTPUT_DIR/f"generation_alpha_{str(last).replace('.','p')}.json"
    rows=json.loads(path.read_text())
    print('Progression stopped at alpha',last,'— showing latest best generation.')

for i,x in enumerate(rows,1):
    print('\n'+'='*100)
    print(i,'USER:',x['prompt'])
    print('QWEN:',x['qwen'])
    print('CeNN-v4:',x['cenn'])
    print('exact=',x['exact'],'jaccard=',round(x['jaccard'],3))


In [ ]:
#@title 8. Inspect on-policy / margin training
if len(hist):
    cols=[
        'alpha','stage_step','train_forward_kl','train_reverse_kl',
        'train_rank_loss','train_margin_loss','train_on_policy_loss','train_on_policy_scale',
        'top1','top1_margin_loss'
    ]
    cols=[c for c in cols if c in hist.columns]
    display(hist[cols].tail(40))

    used=hist[hist['train_on_policy_loss']>0]
    print('On-policy updates recorded:',len(used))
    if len(used):
        display(used[cols].tail(20))


In [ ]:
#@title 9. Upload strict-quality final v4 model to Hugging Face
UPLOAD_TO_HF=False #@param {type:'boolean'}
if not UPLOAD_TO_HF:
    print('Upload skipped. Enable only after reviewing the strict final quality results.')
else:
    from huggingface_hub import HfApi, login
    import shutil, json

    if not report.get('progression_complete'):
        raise RuntimeError('CeNNMixer-v4 has not safely reached alpha=1; HF upload is blocked.')
    if not report.get('strict_quality_gate'):
        raise RuntimeError('CeNN-v4 reached alpha=1 but did not pass the strict quality gate; HF upload is blocked.')

    HF_REPO_ID='vtava/Qwen35-0.8B-CeNNMixer-v4-DeltaCell' #@param {type:'string'}
    HF_PRIVATE=False #@param {type:'boolean'}

    HF_EXPORT=REPO_DIR/'results'/'Qwen35-0.8B-CeNNMixer-v4-DeltaCell-HF'
    if HF_EXPORT.exists():
        shutil.rmtree(HF_EXPORT)
    HF_EXPORT.mkdir(parents=True,exist_ok=True)

    for name in ['cennmixer_v4_final_cenn_only.pt','report.json','stage_summary.csv','training_history.csv']:
        src=OUTPUT_DIR/name
        if src.exists():
            shutil.copy2(src,HF_EXPORT/name)

    (HF_EXPORT/'cennmixer_v4_config.json').write_text(json.dumps({
        'base_model':report['base_model'],
        'architecture':report['architecture'],
        'layers':report['layers'],
        'layer_kinds':report['layer_kinds'],
        'config':report['config'],
        'final_metrics':report['final_cenn_only_probe'],
    },indent=2),encoding='utf-8')

    m=report['final_cenn_only_probe']
    readme=f"""---
    base_model: {report['base_model']}
    library_name: transformers
    pipeline_tag: text-generation
    tags:
    - qwen
    - cenn
    - recurrent
    - delta-rule
    - fast-weight-memory
    - tinycenn
    ---

    # Qwen3.5-0.8B CeNNMixer-v4 DeltaCell

    CeNNMixer-v4 combines CeNN cellular recurrent state with a compact gated-delta
    matrix memory, causal depthwise convolution, gated RMS output, and on-policy
    distillation.

    Replaced layers: {report['layers']}

    Final CeNN-only metrics:
    - Top-1 agreement: {m['top1']:.6f}
    - KL: {m['kl']:.6f}
    - Hidden MSE: {m['hidden_mse']:.6f}
    - Student CE: {m['student_ce']:.6f}
    - Teacher CE: {m['teacher_ce']:.6f}

    Mixer parameter reduction: {report['mixer_param_reduction_pct']:.2f}%

    Project: https://github.com/vtavakkoli/TinyCeNN-LM
    """
    (HF_EXPORT/'README.md').write_text(readme,encoding='utf-8')

    token=None
    try:
        from google.colab import userdata
        token=userdata.get('HF_TOKEN')
    except Exception:
        pass
    if token:
        login(token=token,add_to_git_credential=False)
    else:
        login()

    api=HfApi()
    api.create_repo(HF_REPO_ID,repo_type='model',private=HF_PRIVATE,exist_ok=True)
    api.upload_folder(
        folder_path=str(HF_EXPORT),
        repo_id=HF_REPO_ID,
        repo_type='model',
        commit_message='Upload CeNNMixer-v4 DeltaCell final adapter and results',
    )
    print('✓ Uploaded: https://huggingface.co/'+HF_REPO_ID)
